In [11]:
# ===== Import libraries =====
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score


# ===== Load dataset =====
# ===== Load dataset =====
url = "https://drive.google.com/file/d/1JPFEjSOlQ-gWExGiUv2kB1-yHWGfZdOQ/view?usp=sharing"
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]


housing = pd.read_csv(path)
housing.head()

# ===== Define features and target =====
X = housing.drop(columns=["Id"])
y = np.log1p(X.pop("SalePrice"))   # ✅ log target

# ===== Split data =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== Separate categorical and numerical =====
X_cat = X_train.select_dtypes(include=["object"])
X_num = X_train.select_dtypes(exclude=["object"])

# ===== Preprocessing =====
preprocessor = make_column_transformer(
    (
        make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        ),
        X_cat.columns
    ),
    (
        SimpleImputer(strategy="median"),
        X_num.columns
    ),
    remainder="drop"
)

# ===== Pipeline =====
sfm_pipe = make_pipeline(
    preprocessor,
    SelectFromModel(
        estimator=DecisionTreeRegressor(random_state=42),
        threshold="median"
    ),
    LinearRegression()
)

# ===== Cross-validation (RMSE on log scale) =====
cv_scores = cross_val_score(
    sfm_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

cv_rmse = -cv_scores

print("CV RMSE (log scale):", cv_rmse)
print("Mean CV RMSE (log scale):", cv_rmse.mean())
print("Std CV RMSE:", cv_rmse.std())

# ===== Train model =====
sfm_pipe.fit(X_train, y_train)

# ===== Predict =====
y_pred_log = sfm_pipe.predict(X_test)

# ===== Evaluation (log scale) =====
print("\n--- LOG SCALE ---")
print("Test MAE:", mean_absolute_error(y_test, y_pred_log))
print("Test RMSE:", mean_squared_error(y_test, y_pred_log) ** 0.5)
print("Test MAPE:", mean_absolute_percentage_error(y_test, y_pred_log))
print("Test R2:", r2_score(y_test, y_pred_log))

# ===== Convert back to original scale =====
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

# ===== Evaluation (real price) =====
print("\n--- ORIGINAL PRICE SCALE ---")
print("Test RMSE:", mean_squared_error(y_test_real, y_pred_real) ** 0.5)

# ===== Show selected features =====
feature_names = sfm_pipe.named_steps["columntransformer"].get_feature_names_out()
selected_mask = sfm_pipe.named_steps["selectfrommodel"].get_support()
selected_features = pd.Series(feature_names[selected_mask], name="Selected Features")

print("\nSelected features:")
print(selected_features.to_string(index=False))

CV RMSE (log scale): [0.12703794 0.17627878 0.22878577 0.13357573 0.12537588]
Mean CV RMSE (log scale): 0.15821082011943227
Std CV RMSE: 0.03991001151625251

--- LOG SCALE ---
Test MAE: 0.11226737857273067
Test RMSE: 0.15424232584181066
Test MAPE: 0.009459663290081109
Test R2: 0.8725118211903107

--- ORIGINAL PRICE SCALE ---
Test RMSE: 28568.105266255534

Selected features:
       pipeline__LandContour
      pipeline__Neighborhood
         pipeline__RoofStyle
       pipeline__Exterior1st
         pipeline__ExterQual
        pipeline__Foundation
          pipeline__BsmtQual
      pipeline__BsmtExposure
      pipeline__BsmtFinType1
      pipeline__BsmtFinType2
        pipeline__CentralAir
       pipeline__KitchenQual
        pipeline__Functional
        pipeline__GarageType
      pipeline__GarageFinish
             pipeline__Fence
  simpleimputer__LotFrontage
      simpleimputer__LotArea
  simpleimputer__OverallQual
  simpleimputer__OverallCond
    simpleimputer__YearBuilt
 simpleimputer

#COMPETITION-Kaggle


In [14]:
# ===== Load teacher's external test data =====
url = "https://drive.google.com/file/d/1q14sdW_8Gk9x0h5fAejPpq38xuRwgHnY/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id=" + url.split("/")[-2]

# ===== Load teacher test CSV =====

test_df = pd.read_csv(path)

In [18]:
# ===== Prepare test data =====
# Keep Id column for submission file
test_ids = test_df["Id"]

# Remove Id from features (it is not a real feature)
X_kaggle_test = test_df.drop(columns=["Id"])

# ===== Make predictions (in log scale) =====
test_pred_log = sfm_pipe.predict(X_kaggle_test)
# If you used a different model (e.g., sfm_pipe), use:
# test_pred_log = sfm_pipe.predict(X_kaggle_test)

# ===== Convert predictions back to original SalePrice scale =====
test_pred = np.expm1(test_pred_log)

# ===== Create submission DataFrame =====
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_pred
})

# ===== Save submission file =====
submission.to_csv("submission_SFM_LR.csv", index=False)

# ===== Quick check =====
print(submission.head())
print(submission.columns)

     Id      SalePrice
0  1461  118473.367052
1  1462  151817.774379
2  1463  173579.537434
3  1464  195045.164510
4  1465  198891.809342
Index(['Id', 'SalePrice'], dtype='object')


In [20]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>